In [1]:
import open3d as o3d
import numpy as np
import trimesh
import pymeshlab as ml
from meshlib import mrmeshpy
from tqdm import tqdm

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
dataset = o3d.data.OfficePointClouds()
pcds = []
for pcd_path in dataset.paths:
    pcds.append(o3d.io.read_point_cloud(pcd_path))

selected_pcd = pcds[0]
selected_pcd = selected_pcd.voxel_down_sample(voxel_size=0.01)
selected_pcd, _ = selected_pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
selected_pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
    radius=0.1, max_nn=50))  # increase radius
selected_pcd.orient_normals_consistent_tangent_plane(100)


In [3]:
pcd_tree = o3d.geometry.KDTreeFlann(selected_pcd)
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(selected_pcd, depth=10)[0]
mesh.remove_duplicated_vertices()
mesh.remove_degenerate_triangles()
mesh.remove_unreferenced_vertices()
mesh.remove_non_manifold_edges()

TriangleMesh with 293193 points and 586289 triangles.

In [4]:
o3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

In [5]:
o3d.io.write_triangle_mesh("output_mesh.off", mesh)

True